In [32]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))

HF_ENDPOINT = https://hf-mirror.com


In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [34]:
prompt = "请简要解释大语言模型如何生成文本。"
messages = [{"role": "user", "content": prompt}]

In [35]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,  # transformers 5.x 默认 tokenize=True，必须显式关闭才能得到字符串（"我只要格式化后的文本，分词我自己来"）
    return_dict=False,  # 配合 tokenize=False，确保返回 str 而不是 dict
    add_generation_prompt=True,  #用于控制是否在对话末尾添加Assistant回复的起始标记
    enable_thinking=False  #开启的画会再Assistant起始标记后追加一个空的思考区块（跳过思考）
)
print(text)

<|im_start|>user
请简要解释大语言模型如何生成文本。<|im_end|>
<|im_start|>assistant
<think>

</think>




In [36]:
model_inputs = tokenizer(
    [text],
    return_tensors="pt"
).to(model.device)
print(model_inputs)

{'input_ids': tensor([[151644,    872,    198,  14880,  98237,  30534, 104136,  26288, 102064,
         104949, 100007,  43959, 108704,   1773, 151645,    198, 151644,  77091,
            198, 151667,    271, 151668,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [37]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,  # 开启采样；如果为 False，通常使用贪心搜索
    temperature=0.7,
    top_p=0.8,  # Top-P 核采样，只从累计概率达到 0.8 的候选 token 中采样
    top_k=20,  # Top-K 采样，只保留概率最高的 20 个候选 token
)

print(generated_ids.shape)

torch.Size([1, 157])


In [38]:
input_length = len(model_inputs.input_ids[0])

output_ids = generated_ids[0].tolist()
output_ids = generated_ids[0][input_length:].tolist() #不包含输入

In [39]:
response = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

print(response)

大语言模型（Large Language Model, LLM）通过训练大量文本数据，使其具备理解、生成和生成文本的能力。具体过程包括：

1. **数据训练**：模型通过大量文本数据（如新闻、小说、对话等）进行训练，学习其模式和语义。

2. **参数学习**：模型通过不断优化参数，使其能够更准确地生成符合上下文的文本。

3. **生成文本**：模型在没有明确指令的情况下，根据训练数据生成符合语境的文本，例如写作、回答问题等。

因此，大语言模型的核心是通过训练和学习来生成文本。


# 流式生成

In [40]:
from transformers import TextStreamer

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

print("Assistant: ", end="", flush=True)

model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    streamer=streamer,
)

Assistant: 大语言模型（Large Language Model, LLM）通过训练大量文本数据，使其具备理解和生成自然语言的能力。生成过程通常包括以下几个步骤：

1. **输入文本**：用户输入一段文本或问题。
2. **模型训练**：通过大量数据（如新闻、小说、对话等）训练模型，使其学习语言的结构和模式。
3. **生成文本**：模型基于训练数据和上下文，生成符合语境和逻辑的文本。这可能涉及多种算法，如Transformer架构，用于处理长文本和复杂语义。

模型可以理解用户的问题，生成相关回复，甚至生成全新的文本内容。


tensor([[151644,    872,    198,  14880,  98237,  30534, 104136,  26288, 102064,
         104949, 100007,  43959, 108704,   1773, 151645,    198, 151644,  77091,
            198, 151667,    271, 151668,    271,  26288, 102064, 104949,   9909,
          34253,  11434,   4903,     11,    444,  10994,   7552,  67338, 104034,
         100722, 108704,  20074,   3837, 102989, 102094, 115167,  43959,  99795,
         102064, 104426,   1773,  43959, 100178, 102119, 100630, 116420, 105652,
          48443,     16,     13,   3070,  31196, 108704,    334,   5122,  20002,
          31196, 104383, 108704,  57191,  86119,   8997,     17,     13,   3070,
         104949, 104034,    334,   5122,  67338, 100722,  20074,   9909,  29524,
         100014,   5373, 104032,   5373, 105051,  49567,   7552, 104034, 104949,
           3837, 102989, 100134, 102064,   9370, 100166,  33108, 100144,   8997,
             18,     13,   3070,  43959, 108704,    334,   5122, 104949, 104210,
         104034,  20074,  33

# 手动控制流式输出

In [42]:
from threading import Thread
from transformers import TextIteratorStreamer

streamer = TextIteratorStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

generation_kwargs = {
    **model_inputs,
    "max_new_tokens": 512,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "streamer": streamer,
}

thread = Thread(
    target=model.generate,
    kwargs=generation_kwargs
)

thread.start()

print("Assistant: ", end="", flush=True)

for text_piece in streamer:
    print(text_piece, end="", flush=True)

thread.join()
print()


Assistant: 大语言模型（Large Language Model, LLM）通过深度学习技术，利用大量语言数据训练而成，能够理解并生成文本。其核心原理包括：

1. **数据训练**：模型通过大量文本数据（如书籍、新闻、对话等）进行训练，学习语言的结构和模式。
2. **生成机制**：模型内部使用了复杂的神经网络，可以生成连续的文本，通过训练数据的反馈不断优化生成内容。
3. **推理能力**：模型不仅能生成文本，还能进行推理、理解上下文、生成连贯的段落等。

总之，大语言模型通过深度学习技术，能够理解和生成高质量的文本。
